<a href="https://colab.research.google.com/github/Ayush-Singh-36/Transcribing_Model_PyTorch/blob/main/transcribing_training_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imported the GitHub Repository here

In [9]:
import os
from google.colab import userdata

#retrieve GitHub PAT from colab secrets
github_pat = userdata.get('git_token')

if not github_pat:
  raise ValueError("GITHUB_PAT not found in colab secrets. please add it")

#Original repository URL
repository_url_base = "https://github.com/Ayush-Singh-36/Transcribing_Model_PyTorch.git"

# Constructed the authenticated URL
authenticated_repository_url = repository_url_base.replace("https://github.com", f"https://{github_pat}@github.com/")

print("Authenticated repository URL prepared for cloning.")

Authenticated repository URL prepared for cloning.


# Clone the repository here for model development at this session

In [10]:
repository_url = "https://github.com/Ayush-Singh-36/Transcribing_Model_PyTorch.git"
!git clone {repository_url}
repository_name = repository_url.split("/")[-1].replace(".git", "")
print(f"Repository '{repository_name}' cloned successfully.")
#Using the authenticated URL for cloning
!git clone {authenticated_repository_url}
repository_name = authenticated_repository_url.split("/")[-1].replace(".git", "")
print(f"Repository '{repository_name}'cloned successfully.")

Cloning into 'Transcribing_Model_PyTorch'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 7.82 KiB | 7.82 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Repository 'Transcribing_Model_PyTorch' cloned successfully.
fatal: destination path 'Transcribing_Model_PyTorch' already exists and is not an empty directory.
Repository 'Transcribing_Model_PyTorch'cloned successfully.


# Download the Dataset directly from kaggle to our github directory imported in this notebook

In [11]:
import os
from google.colab import userdata
import sys

def custom_exit(status):
  print(f"Kaggle API tried to exit with status {status}. Ignoring from colab evviornment.")
  #Optionally, raise an exception or log instead of actual exit
sys.exit = custom_exit
exit = custom_exit
try:
  __builtins__.exit = custom_exit

except AttributeError:
  print("Could not patch __builtins__.exit - it might not be present or modifiable in this enviornment.")


#retrieve credentials from colab secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

#now import and run the download code
from kaggle.api.kaggle_api_extended import KaggleApi

dataset_slug = "beomseongkim/torchaudio"
#update download_path to place data within the cloned repository
download_path = "./Transcribing_Model_PyTorch/data"

#create the directory if it doesn't exit
os.makedirs(download_path, exist_ok = True)
print("Authenticating via enviornment variables...")
api = KaggleApi()
api.authenticate()

print("Downloading the dataset from kaggle...")
api.dataset_download_files(dataset_slug, path = download_path, unzip = True)

print(f"Done! Your files have been saved to the '{download_path}' folder.")
#Change directory to the cloned repository
%cd /content/Transcribing_Model_PyTorch

Authenticating via enviornment variables...
Dataset URL: https://www.kaggle.com/datasets/beomseongkim/torchaudio
Done! Your files have been saved to the './Transcribing_Model_PyTorch/data' folder.
/content/Transcribing_Model_PyTorch


# Configure Git remote with Personal Access Token (PAT) for Authentication

In [12]:
import os
from google.colab import userdata
#Retrieve GitHub PAT from google colab secrets
github_pat = userdata.get('git_token')
#We'll re-extract the base URL and repository name
repo_url_base = "https://github.com"
repo_path = repository_url[len(repo_url_base):]
#Construct the authenticated URL
authenticated_repo_url = f"https://{github_pat}@github.com/{repo_path}"
print("Authenticated repository URL created.")

#change to the cloned repository directory if not already there
%cd /content/Transcribing_Model_PyTorch

#set the remote origin to the authenticated URL
!git remote set-url origin (authenticated_repo_url)
#verify the remote URL has been updated
!git remote -v
print("Git remote origin configured with PAT for authentication.")

Authenticated repository URL created.
/content/Transcribing_Model_PyTorch
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `git remote set-url origin (authenticated_repo_url)'
origin	https://github.com/Ayush-Singh-36/Transcribing_Model_PyTorch.git (fetch)
origin	https://github.com/Ayush-Singh-36/Transcribing_Model_PyTorch.git (push)
Git remote origin configured with PAT for authentication.


# Making this dataset comfortable for execution in real-life
*As our Kaggle dataset is lossless audio we can't use it much for training the model for real-life application, that's why we need to noise to our kaggle dataset to make it actually useful*

In [13]:
import torch
import torchaudio
import os
import random

# Define the base data path. Assuming current working directory is /content/Transcribing_Model_PyTorch
base_data_path = "/content/Transcribing_Model_PyTorch/data"
TRAIN_DIR = os.path.join(base_data_path, "train-clean-100/LibriSpeech/train-clean-100")
TEST_DIR = os.path.join(base_data_path, "test-clean/LibriSpeech/test-clean")

# Output directories for noisy data
NOISY_TRAIN_DIR = os.path.join(base_data_path, "train-noisy-100")
NOISY_TEST_DIR = os.path.join(base_data_path, "test-noisy")

# Check if TRAIN_DIR and TEST_DIR exist and have content
if not os.path.exists(TRAIN_DIR) or not os.listdir(TRAIN_DIR):
    print(f"Warning: TRAIN_DIR '{TRAIN_DIR}' is empty or does not exist. Please check your data download and directory structure.")
if not os.path.exists(TEST_DIR) or not os.listdir(TEST_DIR):
    print(f"Warning: TEST_DIR '{TEST_DIR}' is empty or does not exist. Please check your data download and directory structure.")

def add_gaussian_noise(waveform, sample_rate, snr_db=15):
    """
    Adds Gaussian noise to an audio waveform.

    Args:
        waveform (torch.Tensor): The input audio waveform. Shape (channels, samples).
        sample_rate (int): The sample rate of the audio (not directly used for noise generation, but good practice).
        snr_db (float): Desired Signal-to-Noise Ratio in dB.

    Returns:
        torch.Tensor: The waveform with added Gaussian noise.
    """
    # Ensure waveform is float
    if waveform.dtype != torch.float32:
        waveform = waveform.to(torch.float32)

    # Calculate signal power
    # Mean squared value is a common measure for signal power for zero-mean signals
    signal_power = torch.mean(waveform**2)

    if signal_power == 0:
        print("Warning: Signal power is zero. Cannot add noise with specified SNR. Returning original waveform.")
        return waveform

    # Convert SNR from dB to linear scale
    snr_linear = 10**(snr_db / 10)

    # Calculate noise power
    noise_power = signal_power / snr_linear

    # Generate Gaussian noise with calculated power
    # The standard deviation of the noise is sqrt(noise_power)
    noise = torch.randn_like(waveform) * torch.sqrt(noise_power)

    return waveform + noise

def process_audio_directory(input_root_dir, output_root_dir, snr_db=15):
    """
    Processes all audio files in a directory, adds noise, and saves them to an output directory,
    maintaining the original directory structure.

    Args:
        input_root_dir (str): The root directory containing original audio files.
        output_root_dir (str): The root directory where noisy audio files will be saved.
        snr_db (float): Desired Signal-to-Noise Ratio in dB for noise addition.
    """
    os.makedirs(output_root_dir, exist_ok=True)
    print(f"Processing audio files from '{input_root_dir}' to '{output_root_dir}' with SNR {snr_db}dB...")

    processed_count = 0
    skipped_count = 0

    for root, _, files in os.walk(input_root_dir):
        for file in files:
            if file.lower().endswith(('.flac', '.wav', '.mp3')):
                input_filepath = os.path.join(root, file)
                relative_path = os.path.relpath(input_filepath, input_root_dir)
                output_filepath = os.path.join(output_root_dir, relative_path)

                # Create necessary subdirectories in the output path
                os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

                try:
                    waveform, sample_rate = torchaudio.load(input_filepath)
                    noisy_waveform = add_gaussian_noise(waveform, sample_rate, snr_db)
                    torchaudio.save(output_filepath, noisy_waveform, sample_rate)
                    processed_count += 1
                except Exception as e:
                    print(f"Error processing {input_filepath}: {e}")
                    skipped_count += 1
    print(f"Finished processing. Processed {processed_count} files, skipped {skipped_count} files.")


# --- Execute processing for training and testing data ---
# You can change the snr_db value to adjust the noise level

# Process the training data
process_audio_directory(TRAIN_DIR, NOISY_TRAIN_DIR, snr_db=15)

# Process the testing data
process_audio_directory(TEST_DIR, NOISY_TEST_DIR, snr_db=15)

print(f"\nNoisy datasets created in '{NOISY_TRAIN_DIR}' and '{NOISY_TEST_DIR}'.")

Processing audio files from '/content/Transcribing_Model_PyTorch/data/train-clean-100/LibriSpeech/train-clean-100' to '/content/Transcribing_Model_PyTorch/data/train-noisy-100' with SNR 15dB...
Finished processing. Processed 28539 files, skipped 0 files.
Processing audio files from '/content/Transcribing_Model_PyTorch/data/test-clean/LibriSpeech/test-clean' to '/content/Transcribing_Model_PyTorch/data/test-noisy' with SNR 15dB...
Finished processing. Processed 2620 files, skipped 0 files.

Noisy datasets created in '/content/Transcribing_Model_PyTorch/data/train-noisy-100' and '/content/Transcribing_Model_PyTorch/data/test-noisy'.


# Sorting the path of train and test directory

In [16]:
TRAIN_DIR = "/content/Transcribing_Model_PyTorch/data/train-noisy-100"
TEST_DIR = "/content/Transcribing_Model_PyTorch/data/test-noisy"
print("Classes found in train dir:", sorted(os.listdir(TRAIN_DIR)))
print(len(TRAIN_DIR))
print("Classes found in test dir:", sorted(os.listdir(TEST_DIR)))
print(len(TEST_DIR))

Classes found in train dir: ['103', '1034', '1040', '1069', '1081', '1088', '1098', '1116', '118', '1183', '1235', '1246', '125', '1263', '1334', '1355', '1363', '1447', '1455', '150', '1502', '1553', '1578', '1594', '1624', '163', '1723', '1737', '1743', '1841', '1867', '1898', '19', '1926', '196', '1963', '1970', '198', '1992', '200', '2002', '2007', '201', '2092', '211', '2136', '2159', '2182', '2196', '226', '2289', '229', '233', '2384', '2391', '2416', '2436', '248', '250', '2514', '2518', '254', '26', '2691', '27', '2764', '2817', '2836', '2843', '289', '2893', '2910', '2911', '2952', '298', '2989', '302', '307', '311', '3112', '3168', '32', '3214', '322', '3235', '3240', '3242', '3259', '328', '332', '3374', '3436', '3440', '3486', '3526', '3607', '3664', '3699', '3723', '374', '3807', '3830', '3857', '3879', '39', '3947', '3982', '3983', '40', '4014', '4018', '403', '405', '4051', '4088', '412', '4137', '4160', '4195', '4214', '426', '4267', '4297', '4340', '4362', '4397', '440

# Device-Agnostic code to get to know, which hardware are we using

In [17]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda
